# WSD 1 — Preprocessing Data & Kamus KBBI Online

Notebook pertama dari tiga rangkaian WSD untuk esai biologi:

1. **WSD 1 (ini)** — rapikan dataset esai jadi token bersih, lalu ambil semua makna
   kata dari **KBBI online** → kamus kata ambigu
2. **WSD 2** — eksperimen penyempurnaan Lesk (bobot IDF, stopword, uji McNemar)
3. **WSD 3** — versi final: perbandingan guru vs siswa, studi kasus, penandaan warna

Isi notebook ini dua bagian: **A** preprocessing dataset esai, **B** kamus KBBI
online + WSD pertama pada kunci guru.

## Bagian A — Preprocessing dataset esai

Sumber: `Dataset Biologi Transport Membran.xlsx` (sheet *Dataset siswa*):
1.258 jawaban esai dari 34 siswa, 37 soal. Di excel, nama/absen/kelas hanya
ditulis di baris pertama tiap siswa sehingga perlu diisi turun (*forward fill*);
teks jawaban diubah jadi token bersih: huruf kecil, tanpa tanda baca/angka,
tanpa stopword.

In [ ]:
# di Colab jalankan sekali: !pip install Sastrawi -q
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword = StopWordRemoverFactory().create_stop_word_remover()
print("import selesai")

In [ ]:
# Colab: sambungkan Google Drive untuk baca file data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_excel("Dataset Biologi Transport Membran.xlsx", sheet_name="Dataset siswa")
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]   # buang kolom kosong
df = df.dropna(how="all")
df = df.dropna(subset=["Nomer Soal"])

# pilih kolom yang dipakai, lalu isi ke bawah identitas siswa
kolom = [c for c in ["Absen", "Nama", "Kelas", "Nomer Soal", "Jawaban Siswa",
                     "Kunci Guru", "Bobot", "Nilai"] if c in df.columns]
df = df[kolom].copy()
for c in ["Nama", "Absen", "Kelas"]:
    df[c] = df[c].ffill()

print(df.shape, "| siswa unik:", df["Nama"].nunique())
df.head(3)

In [ ]:
def bersihkan(teks):
    """teks -> token bersih: huruf kecil, tanpa tanda baca/angka, tanpa stopword."""
    if pd.isna(teks):
        return []
    teks = str(teks).lower()
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    if teks == "":
        return []
    return stopword.remove(teks).split()

# tes
print(bersihkan("Pompa Na+/K+ menggunakan energi ATP secara langsung!"))

In [ ]:
df["Jawaban_Tokens"] = df["Jawaban Siswa"].apply(bersihkan)
df["Kunci_Tokens"] = df["Kunci Guru"].apply(bersihkan)
df["status_data"] = df["Jawaban_Tokens"].apply(lambda t: "kosong" if len(t) == 0 else "ok")

print("total baris:", len(df), "| jawaban kosong:", (df["status_data"] == "kosong").sum())
print("kolom:", list(df.columns))

In [ ]:
# simpan versi rapi: token digabung lagi jadi teks; kolom Nilai tidak ikut
df_simpan = df[["Absen", "Nama", "Kelas", "Nomer Soal",
                "Jawaban_Tokens", "Kunci_Tokens", "Bobot", "status_data"]].copy()
for c in ["Jawaban_Tokens", "Kunci_Tokens"]:
    df_simpan[c] = df_simpan[c].apply(lambda t: " ".join(t))

df_simpan.to_csv("dataset_biologi_lengkap.csv", index=False, sep=";")
print("tersimpan: dataset_biologi_lengkap.csv |", len(df_simpan), "baris")

## Bagian B — Kamus makna dari KBBI online

Semua token unik di dataset (1.545 kata) dicek satu per satu ke **API KBBI**
`kbbi.raf555.dev` (KBBI VI). Kata dengan **arti > 1** = kata ambigu → masuk
`KAMUS_KBBI` untuk WSD. Setiap jawaban API disimpan ke `kbbi_gloss_cache.csv`
supaya run berikutnya tidak perlu koneksi lagi.

In [ ]:
df = pd.read_csv("dataset_biologi_lengkap.csv", sep=None, engine="python")
df = df.dropna(subset=["Jawaban_Tokens"])

def jadi_list(teks):
    teks = str(teks)
    return teks.split() if teks not in ("", "nan") else []

df["Jawaban_Tokens"] = df["Jawaban_Tokens"].apply(jadi_list)
df["Kunci_Tokens"] = df["Kunci_Tokens"].fillna("").apply(jadi_list)
print(df.shape, "| jawaban kosong:", sum(len(t) == 0 for t in df["Jawaban_Tokens"]))

In [ ]:
import os
import time
import requests

URL_API = "https://kbbi.raf555.dev/api/v1/entry/{kata}"   # KBBI VI (Okt 2025)
FILE_FALLBACK = "kbbi_gloss_lengkap.csv"                  # cadangan offline (opsional)

KBBI_LOKAL = {}
if os.path.exists(FILE_FALLBACK):
    _df = pd.read_csv(FILE_FALLBACK, sep=";")
    KBBI_LOKAL = {str(r["kata"]): str(r["gloss"]).split(" || ")
                  for _, r in _df.iterrows()}
    print("fallback lokal siap:", len(KBBI_LOKAL), "kata")

KBBI_CACHE = {}
STAT_API = 0      # kata yang berhasil dari API online
STAT_LOKAL = 0    # kata yang pakai fallback kalau API gagal

def ambil_gloss(kata):
    """Semua makna satu kata dari KBBI. Utama: API online; darurat: dataset lokal.
    Kata tidak terdaftar di KBBI -> list kosong."""
    global STAT_API, STAT_LOKAL
    kata = str(kata).lower()
    if kata in KBBI_CACHE:
        return KBBI_CACHE[kata]

    gloss = []
    dari_api = False
    try:
        r = requests.get(URL_API.format(kata=kata), timeout=10)
        if r.status_code == 200:
            for entri in r.json().get("entries", []):
                for d in entri.get("definitions", []):
                    teks = (d.get("definition") or "").strip()
                    if teks:
                        gloss.append(teks)
            dari_api = True
        elif r.status_code == 404:
            dari_api = True
    except Exception:
        pass

    if not dari_api:
        gloss = KBBI_LOKAL.get(kata, [])
        STAT_LOKAL += 1
        return gloss

    KBBI_CACHE[kata] = gloss
    STAT_API += 1
    time.sleep(0.05)
    return gloss

print("fungsi ambil_gloss siap (utama: API online, darurat: dataset lokal)")

for kata in ["sel", "bunga", "aliran", "darah"]:
    g = ambil_gloss(kata)
    print(f"{kata} -> {len(g)} makna")
    for n, teks in enumerate(g, 1):
        print(f"   {n}) {teks[:80]}")

In [ ]:
# load cache lama kalau ada, lalu scan semua token unik di dataset
if os.path.exists("kbbi_gloss_cache.csv"):
    _c = pd.read_csv("kbbi_gloss_cache.csv", sep=";")
    for _i in _c.index:
        _k = str(_c.at[_i, "kata"]); _g = _c.at[_i, "gloss"]
        if pd.notna(_g) and str(_g).strip() != "":
            KBBI_CACHE.setdefault(_k, str(_g).split(" || "))
    print("cache dimuat:", len(KBBI_CACHE), "kata")

semua_token = set()
for tokens in df["Jawaban_Tokens"]:
    semua_token.update(tokens)
for tokens in df["Kunci_Tokens"]:
    semua_token.update(tokens)
print("token unik di data:", len(semua_token))

KAMUS_KBBI = {}
sudah = 0
for kata in sorted(semua_token):
    gloss = ambil_gloss(kata)
    if len(gloss) > 1:
        KAMUS_KBBI[kata] = gloss
    sudah += 1
    if sudah % 100 == 0:
        print(f"progress: {sudah}/{len(semua_token)} | ambigu sementara: {len(KAMUS_KBBI)}")

# simpan cache biar run berikutnya nggak panggil API lagi
pd.DataFrame([[k, " || ".join(g)] for k, g in KBBI_CACHE.items()],
             columns=["kata", "gloss"]).to_csv("kbbi_gloss_cache.csv",
                                               index=False, sep=";")
print()
print("kata ambigu (sense > 1) ditemukan:", len(KAMUS_KBBI))
print("contoh:", list(KAMUS_KBBI.keys())[:15])
print(f"sumber: API online = {STAT_API} kata | fallback lokal = {STAT_LOKAL} kata")

In [ ]:
# cek & bersihkan: sebagian gloss KBBI membawa huruf miring unikode (matematis)
import re
import unicodedata
from collections import Counter

census, kata_kena = Counter(), set()
for kata, daftar_gloss in KAMUS_KBBI.items():
    nonascii = [c for c in " || ".join(daftar_gloss) if ord(c) > 127]
    if nonascii:
        kata_kena.add(kata)
        census.update(nonascii)

print(f"{len(kata_kena)} dari {len(KAMUS_KBBI)} kata ambigu mengandung karakter non-ASCII:")
for c, n in census.most_common(5):
    print(f"  U+{ord(c):04X}  {unicodedata.name(c, '?')[:45]:47s} muncul di {n:4d} gloss")

def bersihkan_gloss(teks):
    """normalisasi NFKC + buang sisa karakter CJK."""
    t = unicodedata.normalize("NFKC", str(teks))
    t = re.sub(r"[\u2E80-\u2EFF\u3000-\u303F\u31C0-\u31EF\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]", " ", t)
    return re.sub(r"\s{2,}", " ", t).strip()

n_bersih = 0
for kata in KAMUS_KBBI:
    baru = [bersihkan_gloss(g) for g in KAMUS_KBBI[kata]]
    n_bersih += sum(b != l for b, l in zip(baru, KAMUS_KBBI[kata]))
    KAMUS_KBBI[kata] = baru

# perbarui cache supaya run berikutnya langsung dapat gloss yang bersih
pd.DataFrame([[k, " || ".join(g)] for k, g in KBBI_CACHE.items()],
             columns=["kata", "gloss"]).to_csv("kbbi_gloss_cache.csv",
                                               index=False, sep=";")
print(f">> selesai: {n_bersih} teks gloss dinormalisasi (kamus & cache).")

In [ ]:
# simplified Lesk di atas kamus KBBI online.
# 2 hal penting (hasil uji di ground truth 646 token):
#   1. TANPA stemming -> gloss KBBI pendek, stemming sering merusak kecocokan kata
#      (contoh: "perpindahan" jadi "pindah", tidak cocok lagi dgn konteks)
#   2. kata konteks yang berulang dihitung 1x saja
def lesk(kalimat, kata_target):
    """pilih sense dgn overlap konteks-gloss terbanyak; seri / 0 -> sense pertama."""
    if kata_target not in KAMUS_KBBI:
        return None
    konteks = set(kalimat.split())
    skor = [sum(1 for k in konteks if k != kata_target and k in g.split())
            for g in KAMUS_KBBI[kata_target]]
    return skor.index(max(skor)) + 1

# tes ('bunga' tidak ada di esai biologi -> tidak ada di kamus -> None)
print("sel  :", lesk("sel adalah bagian terkecil makhluk hidup", "sel"))
print("bunga:", lesk("bunga pinjaman di bank naik bulan ini", "bunga"))

In [ ]:
# WSD di kunci guru: 1 soal = 1 kunci, sense tiap kata ambigu disimpan sekali
GURU_SENSE = {}

for soal in df["Nomer Soal"].dropna().unique():
    baris = df[df["Nomer Soal"] == soal].iloc[0]
    kalimat = " ".join(baris["Kunci_Tokens"])
    peta = {}
    for kata in baris["Kunci_Tokens"]:
        if kata in KAMUS_KBBI and kata not in peta:
            peta[kata] = lesk(kalimat, kata)
    GURU_SENSE[soal] = peta

soal_pertama = df["Nomer Soal"].iloc[0]
print("contoh kunci soal", soal_pertama, ":")
for kata, sense in GURU_SENSE[soal_pertama].items():
    print(f"   {kata}/{sense}")
print("soal terproses:", len(GURU_SENSE))

In [ ]:
# bandingkan sense guru vs sense siswa untuk tiap kata ambigu yang sama
baris_tabel = []
sudah = 0
for i in df.index:
    soal = df.at[i, "Nomer Soal"]
    kalimat = " ".join(df.at[i, "Jawaban_Tokens"])
    for kata in df.at[i, "Jawaban_Tokens"]:
        if kata in KAMUS_KBBI and kata in GURU_SENSE[soal]:
            sense_siswa = lesk(kalimat, kata)
            sense_guru = GURU_SENSE[soal][kata]
            ket = "sesuai" if sense_siswa == sense_guru else "tidak sesuai"
            baris_tabel.append([df.at[i, "Nama"], soal, kata,
                                sense_guru, sense_siswa, ket])
    sudah += 1
    if sudah % 100 == 0:
        print("progres:", sudah, "dari", len(df))

df_banding = pd.DataFrame(baris_tabel,
                          columns=["Nama", "Nomer_Soal", "Token",
                                   "Sense_Guru", "Sense_Siswa", "Keterangan"])
print()
print("total pembandingan:", len(df_banding))
print("sesuai      :", (df_banding["Keterangan"] == "sesuai").sum())
print("tidak sesuai:", (df_banding["Keterangan"] != "sesuai").sum())

print()
for i in df_banding.index[:10]:
    b = df_banding.loc[i]
    print(f"jawaban guru: {b['Token']}/{b['Sense_Guru']} | "
          f"jawaban siswa: {b['Token']}/{b['Sense_Siswa']} | {b['Keterangan']}")

In [ ]:
df_banding.to_csv("perbandingan_guru_siswa.csv", index=False, sep=";")
print("tersimpan: perbandingan_guru_siswa.csv ->", len(df_banding), "baris")

# pastikan csv bebas karakter aneh (pakai bersihkan_gloss yang sama)
d = pd.read_csv("perbandingan_guru_siswa.csv", sep=";", dtype=str, keep_default_na=False)
sebelum = d.copy()
for c in d.columns:
    d[c] = d[c].map(bersihkan_gloss)
d.to_csv("perbandingan_guru_siswa.csv", index=False, sep=";")
print("- perbandingan_guru_siswa.csv: pemisah ';',",
      int((d != sebelum).sum().sum()), "sel diperbaiki")

In [ ]:
# daftar kata ambigu di kunci guru -> template ground truth untuk dianotasi manual
baris_gt = []
for soal in sorted(GURU_SENSE.keys()):
    baris = df[df["Nomer Soal"] == soal].iloc[0]
    kalimat = " ".join(baris["Kunci_Tokens"])
    for kata in baris["Kunci_Tokens"]:
        if kata in KAMUS_KBBI:
            daftar = " | ".join(f"{n}) {g}" for n, g in enumerate(KAMUS_KBBI[kata], 1))
            baris_gt.append([soal, kalimat, kata, daftar, "", lesk(kalimat, kata)])

df_gt = pd.DataFrame(baris_gt, columns=["Nomer_Soal", "Kunci_Jawaban",
                                        "kata_token", "Daftar_Gloss",
                                        "Sense_Benar", "hasil_WSD_sistem"])
df_gt = df_gt.drop_duplicates(subset=["Kunci_Jawaban", "kata_token"])
df_gt.to_csv("groundtruth_template.csv", index=False, sep=";")

d = pd.read_csv("groundtruth_template.csv", sep=";", dtype=str, keep_default_na=False)
sebelum = d.copy()
for c in d.columns:
    d[c] = d[c].map(bersihkan_gloss)
d.to_csv("groundtruth_template.csv", index=False, sep=";")

print("baris untuk dianotasi:", len(df_gt))
print("tersimpan: groundtruth_template.csv (kolom Sense_Benar diisi manual)")
print("- groundtruth_template.csv: pemisah ';',",
      int((d != sebelum).sum().sum()), "sel diperbaiki")
df_gt.head(10)

In [ ]:
# evaluasi: bandingkan hasil sistem dgn sense benar hasil anotasi manual
df_gt = None
for enc in ("utf-8", "cp1252", "latin-1"):
    try:
        df_gt = pd.read_csv("groundtruth_selesai.csv", sep=None, engine="python", encoding=enc)
        print("file dibaca dengan encoding:", enc)
        break
    except UnicodeDecodeError:
        continue

df_gt["Sense_Benar"] = pd.to_numeric(df_gt["Sense_Benar"], errors="coerce")
df_gt = df_gt.dropna(subset=["Sense_Benar"]).reset_index(drop=True)

hasil_sistem = [lesk(df_gt.at[i, "Kunci_Jawaban"], df_gt.at[i, "kata_token"])
                for i in df_gt.index]
df_gt["hasil_WSD_sistem"] = hasil_sistem
df_gt["hasil_WSD_asli"] = df_gt["Sense_Benar"].astype(int)
df_gt["cocok"] = df_gt["hasil_WSD_asli"] == df_gt["hasil_WSD_sistem"]

tabel_final = df_gt[["Kunci_Jawaban", "kata_token",
                     "hasil_WSD_asli", "hasil_WSD_sistem", "cocok"]]
benar = df_gt["cocok"].sum()
print(f"Accuracy (vs ground truth): {benar}/{len(df_gt)} = {round(df_gt['cocok'].mean(), 3)}")
tabel_final.to_csv("hasil_groundtruth.csv", index=False, sep=";")
print("tersimpan: hasil_groundtruth.csv")
tabel_final.head(15)

## Hasil notebook ini & lanjutan

| file keluaran | isi |
|---|---|
| `dataset_biologi_lengkap.csv` | esai rapi, 1.258 baris token bersih |
| `kbbi_gloss_cache.csv` | cache makna 1.546 kata dari API KBBI |
| `perbandingan_guru_siswa.csv` | 13.955 perbandingan sense, 81,1% sesuai |
| `groundtruth_template.csv` → `groundtruth_selesai.csv` | 646 kata ambigu di kunci guru, dianotasi manual |
| `hasil_groundtruth.csv` | evaluasi sistem vs ground truth |

Akurasi Lesk versi ini: **395/646 = 61,1%** → disempurnakan di **WSD 2**
(bobot IDF, buang stopword, uji McNemar).